# 2D Likelihood Contour: GW Strain vs Pulsar Distance

This notebook generates **N** random pulsars and **M** random continuous
gravitational wave (CGW) sources, then sweeps the PTA log-likelihood over
a 2D grid of:

- **GW strain amplitude** ($\log_{10} h$) of one CW source (x-axis)
- **Pulsar distance** (PX, in kpc) of one target pulsar (y-axis)

The result is a contour plot showing how the likelihood constrains these
two parameters jointly.

In [1]:
%matplotlib widget
import matplotlib.pyplot as plt

In [ ]:
from __future__ import annotations

from loguru import logger
logger.disable("pint")

from io import StringIO

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

import pint.models as pm

from jaxpint.pta.likelihood import pta_logL
from jaxpint.notebook_utils import (
    build_cw_injectors,
    generate_random_par,
    inject_and_build_config,
    plot_2d_delta_logL,
    setup_synthetic_pta,
    sweep_1d_logL,
    sweep_2d_logL,
)

# ---- Configuration ----
N_PULSARS = 10
M_CW_SOURCES = 1
N_TOAS = 200
START_MJD = 57000.0
END_MJD = 60000.0       # ~8 yr observation span
TOA_ERROR = 1e-8         # 10 ns
FREQ = 1400.0            # MHz
SEED = 42

## Generate random pulsars

Each pulsar gets a random sky position, spin frequency, spindown rate, and
distance. The par file includes spindown (F0, F1), astrometry (RAJ, DECJ, PX),
and unscaled white noise (EFAC = 1).

In [ ]:
rng = np.random.default_rng(SEED)

par_strings = [generate_random_par(idx, rng, start_mjd=START_MJD) for idx in range(N_PULSARS)]
pint_models = [pm.get_model(StringIO(p)) for p in par_strings]

print(f"Generated {N_PULSARS} pulsars")
print(f"Example .par:\n{par_strings[0]}")


## Generate fake TOAs and convert to JaxPINT

In [ ]:
synthetic = setup_synthetic_pta(
    pint_models,
    start_mjd=START_MJD, end_mjd=END_MJD,
    n_toas=N_TOAS, toa_error_s=TOA_ERROR, freq_mhz=FREQ,
)
pp_tuple = synthetic.pulsar_params_list

for i, model in enumerate(pint_models):
    px_val = float(pp_tuple[i].param_value("PX"))
    f0 = float(pp_tuple[i].param_value("F0"))
    print(f"  Pulsar {i}: {model.PSR.value:>20s}  PX(dist)={px_val:.2f} kpc  F0={f0:.1f} Hz")

print(f"\nAll {N_PULSARS} pulsars loaded.")


## Set up M CW sources and inject into TOAs

We place M continuous gravitational wave sources at random sky locations with
random nHz-band GW frequencies and strain amplitude $h = 10^{-12}$.

In [ ]:
TRUE_LOG10_H = -14.0

cw_injectors, _ = build_cw_injectors(
    pint_models, n_sources=M_CW_SOURCES, rng=rng, log10_h=-14.0,
)

for m, inj in enumerate(cw_injectors):
    print(
        f"  CW source {m}: cos_gwtheta={inj.param_spec['cos_gwtheta']:.3f}, "
        f"gwphi={inj.param_spec['gwphi']:.3f}, "
        f"log10_fgw={inj.param_spec['log10_fgw']:.2f}"
    )

gp, config = inject_and_build_config(synthetic, cw_injectors)

# Capture the true injected value for source 0 if the swept param is frequency
if "log10_h" == "log10_fgw":
    TRUE_LOG10_H = float(cw_injectors[0].param_spec["log10_fgw"])

print(f"\nPTA config built with {M_CW_SOURCES} CW sources.")
print(f"Global params: {gp.names}")


## 2D likelihood sweep: log10(h) vs pulsar distance

We sweep the strain amplitude of CW source 0 (`cw0_log10_h`) and the
distance of pulsar 0 (`PX`) over a 2D grid, evaluating the PTA
log-likelihood at each point.

In [ ]:
TARGET_PULSAR = 0
true_distance = float(pp_tuple[TARGET_PULSAR].param_value("PX"))
print(f"Pulsar {TARGET_PULSAR} true distance: {true_distance:.3f} kpc")
print(f"True log10(h) for CW source 0: {TRUE_LOG10_H}")

half_window_px = 0.005
log10_h_grid = np.linspace(-14.72, -13.8, 500)
distance_grid = np.linspace(
    true_distance - half_window_px,
    true_distance + half_window_px,
    500,
)

def eval_logL_2d(log10_h_val, px_val):
    gp_mod = gp.with_value("cw0_log10_h", log10_h_val)
    pp_mod_0 = pp_tuple[TARGET_PULSAR].with_value("PX", px_val)
    pp_mod = pp_tuple[:TARGET_PULSAR] + (pp_mod_0,) + pp_tuple[TARGET_PULSAR + 1:]
    return pta_logL(gp_mod, pp_mod, config)

print(f"Computing {len(log10_h_grid)} x {len(distance_grid)} = "
      f"{len(log10_h_grid) * len(distance_grid)} likelihood evaluations...")
logL_2d = sweep_2d_logL(eval_logL_2d, log10_h_grid, distance_grid)
print("Done.")


## Contour plot

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

delta_2d = logL_2d - logL_2d.max()
delta_2d = np.clip(delta_2d, -500, 0)

cf = ax.contourf(log10_h_grid, distance_grid, delta_2d, levels=30, cmap="viridis")
ax.contour(
    log10_h_grid, distance_grid, delta_2d,
    levels=[-100, -50, -20, -10, -5, -1],
    colors="white", linewidths=0.8, linestyles="--",
)

ax.plot(TRUE_LOG10_H, true_distance, "r*", markersize=15, label="True values")

ax.set_xlabel(r"$\log_{10}(h)$", fontsize=13)
ax.set_ylabel("Pulsar distance (kpc)", fontsize=13)
ax.set_title(
    f"PTA likelihood: GW strain vs pulsar distance\n"
    f"({N_PULSARS} pulsars, {M_CW_SOURCES} CW sources, "
    f"sweeping source 0 strain & pulsar {TARGET_PULSAR} distance)",
    fontsize=13,
)
ax.legend(fontsize=12, loc="upper left")
ax.tick_params(labelsize=11)
fig.colorbar(cf, ax=ax, label=r"$\Delta$ log-likelihood")
fig.tight_layout()
plt.show()
